# Summaries Analysis

This notebook performs both **quantitative** and **qualitative** analyses of the summaries generated by the API, to build a complete picture of model summary performance.





**1. Quantitative Analysis**
- Conduct a **statistical and exploratory data analysis** of the API results using Pandas and descriptive statistics.  
- Explore and refine the **visualizations** to better highlight differences and distributions.  
- Compare **model runtimes** (CPU vs GPU) for the same document to assess performance efficiency.  



**1. Qualitative Analysis**
- **Garbage Detection:** Identify low-quality or nonsensical summaries (e.g., hallucinations or “garbage” outputs).  
  These can often be detected as **outliers** in the quantitative performance metrics.  
  DeepSeek has shown clear examples of this behavior.  
- **Summary Comparison:** Evaluate the **relative quality and performance** of summaries across different models, linking the findings with the quantitative metrics.


## Quantitative Analysis

#### Load data and function definitions

In [ ]:
# Libraries imports

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Function definitions

In [ ]:
# Attempt to identify parameter scale (in billions) from model names
def get_model_parameter_scale(model_name):

    import re
    if not model_name:
        return None

    known_params = {
        "gemma3:12b": 12,
        "gemma3:40b": 40,
        "gemma3:20b": 20,
        "gemma3:22b": 22,
        "gemma3:38b": 38,
        "llama3.1:8b": 8,
        "llama3.2:3b": 3,
        "llama3.2:11b": 11,
        "deepseek-r1:1.1b": 1.1,
        "deepseek-r1:1b": 1,
        "phi-3:3.8b": 3.8,
        "phi-4:14b": 14,
        "qwen3:3.8b": 3.8,
        "qwen3:14b": 14,
        "qwen3:72b": 72,
    }

    model_name_lower = model_name.lower()
    if model_name_lower in known_params:
        return known_params[model_name_lower]

    match = re.search(r"(\d+(?:\.\d+)?)\s*b", model_name_lower)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None

def unique_vals(df, col):
    print(col)
    return print(f"Total {col}: {(df[col].nunique())} \n {(df[col].unique())} \n")


In [ ]:
DATA_PATH = '/Users/sofi/Desktop/collectiveAI/'
cuda_summaries_file = os.path.join(DATA_PATH, 'summarization-benchmark-results-cuda.json')
cpu_summaries_file = os.path.join(DATA_PATH, 'summarization-benchmark-results-cpu.json')

with open(cuda_summaries_file, 'r') as f:
    cuda_summaries = json.load(f)
with open(cpu_summaries_file, 'r') as f:
    cpu_summaries = json.load(f)

summaries = cuda_summaries + cpu_summaries
df = pd.DataFrame(summaries)

In [ ]:
df.doc_path.nunique()

In [ ]:
df.doc_path = df.doc_path.apply(lambda x: os.path.basename(x))
df.doc_path.nunique()

### Descriptive analysis

In [ ]:
df.head(4)

#### Prompts visualization/print

In [ ]:
import pprint
unique = df[['system_prompt','system_prompt_type']].drop_duplicates().reset_index(drop=True)

for _, row in unique.iterrows():
    print("Type:", row['system_prompt_type'])
    print("Prompt:")
    pprint.pprint(row['system_prompt'])
    print("---\n")

In [ ]:
df.info()

#### Data engineer (ms -> mins)

Create new columns to mins unit

In [ ]:
ms_cols = [c for c in df.columns if c.endswith('ms')]
ms_min_cols = [[c.replace('ms', 'min'), c] for c in ms_cols]
time_cols = [c[0] for c in ms_min_cols]

for cols in ms_min_cols:
    c, ms_col = cols
    df[c] = df[ms_col] / 1000 / 60

df.head(2)

time_cols

#### Describe columns

We describe the float type columns:

In [ ]:
desc = df[['input_tokens', 'output_tokens', 'total_tokens', 
           'input_duration_min', 'output_duration_min', 
           'model_duration_min', 'tokens_per_second']].describe().T
print(desc)

In [ ]:
840 / 60

We observed that the 75% of the results have model_duration_min <= 7.7 minutes, and the is at list one outlier with model_duration_min = 840 minutes (14 hours)

In [ ]:
df[df['model_duration_min']>840]

This is an outlier due the garbage of its output:

In [ ]:
import pprint
df[df['model_duration_min']>840].chat_response.iloc[0][-200:]

Visualization of unique values per main columns:

In [ ]:
for col in ['model','system_prompt','device','system_prompt_type','user_prompt','doc_path']:
    unique_vals(df,col)

### Model comparison

In [ ]:
df.groupby('model')[['tokens_per_second', 'model_duration_min', 'total_tokens']].mean().sort_values('tokens_per_second', ascending=False)
df.groupby(['model', 'device'])[['model_duration_min', 'tokens_per_second']].mean()

In [ ]:
# Runtime distribution
plt.figure(figsize=(8,5),dpi = 50)
sns.boxplot(data=df, x='model', y='model_duration_min', hue='device')
plt.xticks(rotation=45)
plt.title("Model Duration by Device")
plt.grid(True) 
plt.show()

Since deepseek-r1:8b has a duration one or two orders of magnitude longer compared to other models, we removed it from the visualization to avoid scaling issues:

In [ ]:
df_to_plot = df[df['model'] != 'deepseek-r1:8b']
# Runtime distribution
plt.figure(figsize=(8,5),dpi = 100)
sns.boxplot(data=df_to_plot, x='model', y='tokens_per_second', hue='device')
plt.xticks(rotation=45)
plt.title("Tokens per second by device")
plt.grid(True) 
plt.show()

In [ ]:
df_to_plot = df[df['model'] != 'deepseek-r1:8b']
# Runtime distribution
plt.figure(figsize=(8,5),dpi = 100)
sns.boxplot(data=df_to_plot, x='model', y='tokens_per_second', hue='system_prompt_type')
plt.xticks(rotation=45)
plt.title("Model Duration by device")
plt.grid(True) 
plt.show()

In [ ]:
df_to_plot = df[df['model'] != 'deepseek-r1:8b']
# Runtime distribution
plt.figure(figsize=(8,5),dpi = 100)
sns.boxplot(data=df_to_plot, x='model', y='model_duration_min', hue='system_prompt_type')
plt.xticks(rotation=45)
plt.title("Model Duration by device")
plt.grid(True) 
plt.show()

It can be seen how much better cuda performs over cpu

In [ ]:
# Throughput
plt.figure(figsize=(8,5),dpi = 100)
sns.barplot(data=df_to_plot, x='model', y='tokens_per_second', hue='system_prompt_type')
plt.xticks(rotation=45)
plt.grid(True) 
plt.title("Throughput (tokens/sec) by Model and Prompt Type")
plt.show()

In [ ]:
# Throughput
plt.figure(figsize=(8,5),dpi = 100)
sns.barplot(data=df_to_plot, x='model', y='model_duration_min', hue='system_prompt_type')
plt.xticks(rotation=45)
plt.grid(True) 
plt.ylabel('Model duration (min)')
plt.title("Tokens per second by Model and Prompt Type")
plt.show()

In [ ]:
# Token ratios
plt.figure(figsize=(8,5),dpi = 100)
df_to_plot['compression_ratio'] = df_to_plot['output_tokens'] / df['input_tokens']
sns.violinplot(data=df_to_plot, x='model', y='compression_ratio')
plt.xticks(rotation=45)
plt.grid(True) 
plt.title("Output/Input Token Ratio (Compression Quality Proxy)")
plt.show()


A nivel 

The mean imput tokens by model are:

In [ ]:
for filter, df_filtered in [("all", df), ("cuda", df[df['device']=='cuda']), ("cpu", df[df['device']=='cpu'])]:
    fig, axes = plt.subplots(ncols=2, figsize=(10, 5))
    label = filter if filter != "all" else filter + " devices"

    sns.barplot(
        data=df_filtered,
        x="model",
        y="tokens_per_second",
        ax=axes[0],
        order=vc.index if 'vc' in globals() else None, label = label
    )
    axes[0].set_title("Generation Speed by Model")
    axes[0].set_ylabel("Tokens per Second")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)

    sns.barplot(
        data=df_filtered,
        x="model",
        y="measured_duration_min",
        ax=axes[1],
        order=vc.index if 'vc' in globals() else None, label = label
    )
    axes[1].set_title("Total Processing Time by Model")
    axes[1].set_ylabel("Duration (min)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()


### Gap Analysis (CPU - CUDA)

#### Time Gap

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()
pivot.head()

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()
if {'cpu', 'cuda'}.issubset(pivot.columns):
    # ensure no missing values and work on a copy
    pivot = pivot.reindex(pivot.index).fillna(0).copy()

    # reset index so 'model' becomes a column
    pivot = pivot.reset_index().rename(columns={'index': 'model'})

    # ensure numeric types and compute difference (values are already in minutes)
    pivot['cpu'] = pd.to_numeric(pivot['cpu'], errors='coerce').fillna(0)
    pivot['cuda'] = pd.to_numeric(pivot['cuda'], errors='coerce').fillna(0)
    pivot['cpu_minus_cuda_min'] = pivot['cpu'] - pivot['cuda']

    # build a 1-D list of model names ordered by the gap
    order = list(pivot.sort_values('cpu_minus_cuda_min', ascending=False)['model'])

    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=pivot,
        x='model',
        y='cpu_minus_cuda_min',
        order=order
    )
    plt.axhline(0, color='k', linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45)
    plt.ylabel('CPU - CUDA duration (min)')
    plt.title('Time gap between CPU and CUDA by model (CPU - CUDA)')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to compute CPU vs CUDA gap (missing 'cpu' or 'cuda' device rows).")

#### Token per second gap

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
if {'cpu', 'cuda'}.issubset(pivot.columns):
    # ensure no missing values and work on a copy
    pivot = pivot.reindex(pivot.index).fillna(0).copy()

    # reset index so 'model' becomes a column
    pivot = pivot.reset_index().rename(columns={'index': 'model'})

    # ensure numeric types and compute difference (values are already in minutes)
    pivot['cpu'] = pd.to_numeric(pivot['cpu'], errors='coerce').fillna(0)
    pivot['cuda'] = pd.to_numeric(pivot['cuda'], errors='coerce').fillna(0)
    pivot['cpu_minus_cuda_min'] = pivot['cpu'] - pivot['cuda']

    # build a 1-D list of model names ordered by the gap
    order = list(pivot.sort_values('cpu_minus_cuda_min', ascending=False)['model'])

    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=pivot,
        x='model',
        y='cpu_minus_cuda_min',
        order=order
    )
    plt.axhline(0, color='k', linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45)
    plt.ylabel('CPU - CUDA tokens per second gap')
    plt.title('Token per second gap between CPU and CUDA by model (CPU - CUDA)')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to compute CPU vs CUDA gap (missing 'cpu' or 'cuda' device rows).")

In [ ]:
df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()

In [ ]:

pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

# prepare a list of models to keep consistent ordering (try to reuse vc order if present)
models = list(vc.index) if 'vc' in globals() else sorted(df['model'].unique())

# build a safe summary even if one device is missing
def safe_gap(pivot, models):
    # ensure index contains all models and missing values filled with 0
    p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
    # coerce numeric and add cpu/cuda columns if missing
    p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
    p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
    return p

p_token = safe_gap(pivot_token, models)
p_dur = safe_gap(pivot_dur, models)

model_summary = pd.DataFrame({
    'model': p_token['model'],
    'gap_input_tokens': p_token['cpu'] - p_token['cuda'],
    # convert minutes gap to seconds
    'gap_total_duration_min': (p_dur['cpu'] - p_dur['cuda']),
})

# bubble size: proportional to number of samples per model (scaled for plotting)
counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50

# ensure columns have expected dtypes
model_summary['gap_input_tokens'] = pd.to_numeric(model_summary['gap_input_tokens'], errors='coerce').fillna(0)
model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

model_summary.head()


In [ ]:

pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

# prepare a list of models to keep consistent ordering (try to reuse vc order if present)
models = list(vc.index) if 'vc' in globals() else sorted(df['model'].unique())

# build a safe summary even if one device is missing
def safe_gap(pivot, models):
    # ensure index contains all models and missing values filled with 0
    p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
    # coerce numeric and add cpu/cuda columns if missing
    p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
    p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
    return p

p_token = safe_gap(pivot_token, models)
p_dur = safe_gap(pivot_dur, models)

model_summary = pd.DataFrame({
    'model': p_token['model'],
    'gap_tokens': p_token['cpu'] - p_token['cuda'],
    # convert minutes gap to seconds
    'gap_total_duration_min': (p_dur['cpu'] - p_dur['cuda']),
})

# bubble size: proportional to number of samples per model (scaled for plotting)
counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50

# ensure columns have expected dtypes
model_summary['gap_tokens'] = pd.to_numeric(model_summary['gap_tokens'], errors='coerce').fillna(0)
model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

model_summary.head()


fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    model_summary["gap_tokens"],
    model_summary["gap_total_duration_min"],
    #s=model_summary["bubble_size"],
    alpha=0.7,
    color="#1f77b4",
    edgecolors="black",
)
for _, row in model_summary.iterrows():
    ax.text(
        row["gap_tokens"],
        row["gap_total_duration_min"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=0.9,
    )
ax.set_ylabel("Average Total Duration Gap (min)")
ax.set_xlabel("Average Tokens per second")
#ax.set_ylabel("Average Total Duration (s)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CPU vs CUDA Performance Gap by Model")
plt.show()

In [ ]:
def make_summary(df):
    pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
    pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

    # prepare a list of models to keep consistent ordering
    models = sorted(df['model'].unique())

    # build a safe summary even if one device is missing
    def safe_gap(pivot, models):
        p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
        p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
        p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
        return p

    p_token = safe_gap(pivot_token, models)
    p_dur = safe_gap(pivot_dur, models)

    model_summary = pd.DataFrame({
        'model': p_token['model'],
        # CUDA - CPU gap for tokens per second
        'gap_tokens': p_token['cuda'] - p_token['cpu'],
        # CUDA - CPU gap for total duration (minutes)
        'gap_total_duration_min': p_dur['cuda'] - p_dur['cpu'],
    })

    # add parameter scale and bubble size
    model_summary['param_scale'] = model_summary['model'].apply(get_model_parameter_scale)
    model_summary['param_scale'] = pd.to_numeric(model_summary['param_scale'], errors='coerce').fillna(0)

    #counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
    #model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50

    # ensure numeric types and no NaNs
    model_summary['gap_tokens'] = pd.to_numeric(model_summary['gap_tokens'], errors='coerce').fillna(0)
    model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

    return model_summary

model_summary = make_summary(df)
model_summary.head()

model_summary.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# generate a unique color per model
palette = sns.color_palette( n_colors=len(model_summary))
model_summary.sort_values("param_scale", inplace=True)
model_colors = dict(zip(model_summary["model"], palette))

fig, ax = plt.subplots(figsize=(12, 6))
i = 1
# plot bubbles with model-specific colors
for _, row in model_summary.iterrows():
    color = model_colors[row["model"]]
    ax.scatter(
        
        row["param_scale"], #row["gap_total_duration_min"],
        row["gap_tokens"],
        s=row["param_scale"] * 100,
        alpha=0.7,
        color=color,
        edgecolors="black",
        linewidth=0.7,
    )
    ax.text(
        row["param_scale"] + (0.5*(-1)**i), #row["gap_total_duration_min"]
        row["gap_tokens"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=1,
        fontweight='bold',
        color=color,
    )
    i += 1
    
    # add a label for the bubble size

ax.set_ylabel("Average Tokens per Second")
ax.set_xlabel("Parameter Scale (Billion Parameters)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CUDA - CPU Gap Token per Second by Model Parameter Scale per Model")

# optional legend showing color per model
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=model,
               markerfacecolor=color, markersize=8, markeredgecolor="black")
    for model, color in model_colors.items()
]
ax.legend(handles=handles, title="Model", loc="best")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# generate a unique color per model
palette = sns.color_palette( n_colors=len(model_summary))
model_summary.sort_values("param_scale", inplace=True)
model_colors = dict(zip(model_summary["model"], palette))

fig, ax = plt.subplots(figsize=(12, 6),dpi = 100)
i = 1
# plot bubbles with model-specific colors
for _, row in model_summary.iterrows():
    color = model_colors[row["model"]]
    ax.scatter(
        
        row["param_scale"], #row["gap_total_duration_min"],
        row["gap_total_duration_min"],
        s=row["param_scale"] * 100,
        alpha=0.7,
        color=color,
        edgecolors="black",
        linewidth=0.7,
    )
    ax.text(
        row["param_scale"] + (0.5*(-1)**i), #row["gap_total_duration_min"]
        row["gap_total_duration_min"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=1,
        fontweight='bold',
        color=color,
    )
    i += 1
    
    # add a label for the bubble size

ax.set_ylabel("Average Total Duration Gap (min)")
ax.set_xlabel("Parameter Scale (Billion Parameters)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CUDA - CPU Gap Token per Second by Model Parameter Scale per Model")

# optional legend showing color per model
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=model,
               markerfacecolor=color, markersize=8, markeredgecolor="black")
    for model, color in model_colors.items()
]
ax.legend(handles=handles, title="Model", loc="best")

plt.tight_layout()
plt.show()


In [ ]:
(df.iloc[0].output_tokens / df.iloc[0].output_duration_ms)*1000 # it is tokens_per_second

In [ ]:
from aymurai.utils.json_data import load_json, save_json

save_json(summaries,DATA_PATH + "summaries.json", )

In [ ]:
df.iloc[0].input_duration_ms + df.iloc[0].output_duration_ms   # total duration in ms

## Qualitative Analysis

### Garbage Detection - Outliers 

In [ ]:
df.describe()

max output_tokens is 163840, same value as max input token, so, it is truncated. We will see examples

In [ ]:
df_truncated = df[df['output_tokens']>16000]
print('Truncated summary analysis: \ndoc paths: \n',df_truncated.doc_path.unique())
print('Models: \n',df_truncated.model.unique())
df_truncated.head()

In [ ]:
df_truncated.groupby(['model','doc_path','system_prompt_type']).count()

11 of 12 outputs truncated correspond to deepseek-r1:8b model, and the other one to phi3:3.8b. We will see the hallucinations bellow

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df, x='model', y='model_duration_ms')
plt.xticks(rotation=45)
plt.title("Runtime Variability Across Documents")
plt.ylabel("Model Duration (ms)")
plt.grid(True, alpha=0.4)
plt.show()


In [ ]:
corr = df_to_plot[['input_tokens','output_tokens','model_duration_ms','tokens_per_second','compression_ratio']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Between Metrics")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_to_plot, x='model', y='tokens_per_second',hue='device')
plt.xticks(rotation=45)
plt.title("Tokens per Second Variability Across Documents")
plt.ylabel("Tokens per Second")
plt.grid(True, alpha=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_to_plot, x='model', y='model_duration_min')
plt.xticks(rotation=45)
plt.title("Runtime Variability Across Documents")
plt.ylabel("Model Duration (min)")
plt.grid(True, alpha=0.4)
plt.show()


### Analysis by document

In [ ]:
df_cuda = df[df['device']=='cuda']
plt.figure(figsize=(10,6))
sns.boxplot(data=df_cuda, x='doc_path', y='tokens_per_second', hue='model')
plt.xticks(rotation=90)
plt.title("Tokens per second by Document and Model - CUDA")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='doc_path', y='tokens_per_second', hue='model')
plt.xticks(rotation=90)
plt.title("Throughput by Document and Model")
plt.tight_layout()
plt.show()


In [ ]:
order = sorted(df['model'].unique())
order

In [ ]:
i = 1
for doc in df['doc_path'].unique():
    df_doc = df[df['doc_path'] == doc]
    doc_name = doc.split('/')[-1]
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df_doc, x='model', y='tokens_per_second', hue='device', order=order)
    plt.xticks(rotation=90)
    plt.grid(True)
    plt.title(f"Fig. {i}: {doc_name}")
    plt.tight_layout()
    i += 1
    plt.show()

#### Summary comparison (same document)

##### cuda vs llama (same model)

In [ ]:
last_N = 1000
document =  '1- FLORES ABARCA, Francisco Alexander s 149 bis J-01-00369422-0-2022-1 Sala Feria (II) nnya vict do apela pp.pdf'
model = 'llama3.2:3b'
devices = ['cuda','cpu']
df_doc = df[df['doc_path']==document]
for d in devices:
    df_filtered = df_doc[(df['model'] == model) & (df['device']==d) & (df['system_prompt_type']=='template')]
    print(f"--------------------- Document: {document}, device= {d}, model= {model}---------------------")
    text = df_filtered['chat_response'].astype(str).fillna('')[-last_N:].values[0]
    print(text)
    print("\n")

  **Interpretation**
- Running the same model (Llama 3.2:3b) and same prompt/document in cuda is  5–6× faster than in cpu
- Despite the speedup, output content remains consistent, with both versions extracting similar entities and structure.
- The CUDA output finishes much faster but tends to include slightly more redundant or verbose entities (e.g., repeated names), suggesting minor decoding variability due to parallelization.
- The CPU output is slower but more stable and concise, with slightly better formatting consistency.

This comparison confirms that CUDA significantly improves throughput without major semantic drift, though some token-level differences may appear.

##### gemma vs llama (cuda)

In [ ]:
last_N = 1000
document = 'aymurai - ejemplo 02.docx'
models = ['phi3:3.8b','gemma3:12b']
df_doc = df[df['doc_path']==document]
for m in models:
    df_filtered = df_doc[(df['model'] == m) & (df['device']=='cuda') & (df['system_prompt_type']=='template')]
    print(f"--------------------- Document: {document}, model= {m}---------------------")
    text = df_filtered['chat_response'].astype(str).fillna('')[-last_N:].values[0]
    print(text)
    print("\n")

- Both models now generate coherent, well-structured summaries capturing key actors, context, and applied laws.
Phi 3:3.8b stands out for its richer narrative detail , like “Ana Carolina Rodríguez (DNI 34.112.456) y su hija menor M.F.R”

- Gemma 3:12b produces a clearer and more factual summary, focusing on explicit information and structured entities.
- Phi offers greater completeness and contextual depth, while Gemma achieves higher precision and speed.
Overall, they show complementary strengths: Phi excels in descriptive depth, Gemma in concise factual extraction.

##### Same document, phi3 vs. phi4

In [ ]:
df.columns

In [ ]:
set(df.model)

In [ ]:
set(df.doc_path)

In [ ]:
df.columns

In [ ]:
last_N = 1000
document = 'aymurai - ejemplo 02.docx'
models = ['phi3:3.8b','phi4:14b']
df_doc = df[df['doc_path']==document]
for m in models:
    df_filtered = df_doc[(df['model'] == m) & (df['device']=='cuda') & (df['system_prompt_type']=='template')]
    print(f"--------------------- Document: {document}, model= {m}---------------------")
    text = df_filtered['chat_response'].astype(str).fillna('')[-last_N:].values[0]
    print(text)
    print("\n")

When comparing phi3:3.8b and phi4:14b on the same document, the improvement in phi4 is evident both in structure and content quality. 

While phi3 tends to generate verbose outputs and include unnecessary placeholders (e.g., “not mentioned in the document provided”), phi4 produces concise, relevant, and well-organized information. 

Its summaries remain focused on the extracted entities without overgenerating text, which improves readability and coherence. The token usage, shown in Fig. 4, also confirm this efficiency: phi4 requires fewer output tokens while maintaining higher generation speed and precision in content extraction.

#### Tokens/sec by document: cuda and cpu

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Compute mean and std per document/model/device
df_summary = (
    df.groupby(['doc_path', 'model', 'device'])['tokens_per_second']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
df_summary['sem'] = df_summary['std'] / df_summary['count']**0.5  # standard error

# Sort models for consistent order
#order = sorted(df_summary['model'].unique())

# --- Plot for each device type ---
for device in ['cuda', 'cpu']:
    df_device = df_summary[df_summary['device'] == device]

    plt.figure(figsize=(10,6))
    
    # Plot one line per document
    for doc, df_doc in df_device.groupby('doc_path'):
        doc_name = doc.split('/')[-1][:20]
        plt.errorbar(
            df_doc['model'], df_doc['mean'],
            yerr=df_doc['sem'], fmt='-o',
            capsize=3, alpha=0.7, label=doc_name
        )
    
    plt.xticks(rotation=90)
    plt.xlabel("Model")
    plt.ylabel("Tokens per second (mean ± SEM)")
    plt.title(f"Performance across documents ({device.upper()})")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(title="Document", fontsize='small', loc = 'best')# bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


**Overall speed:**
- On CUDA, generation speed ranges from ~10 to 90 tokens/sec — up to 10× faster than CPU.
- On CPU, all models stay below 13 tokens/sec, showing clear hardware dependency.
  
**Model comparison:**
- Llama 3.2:3b consistently achieves the highest speed across both devices, though its summary quality is limited.
- Phi 3/4 and DeepSeek 1.1b are among the slowest, likely due to heavier contextual reasoning or token complexity — however, DeepSeek often fails to summarize properly, repeating or looping.
- Gemma 3:12b and Gemma 3:4b show balanced performance, maintaining moderate speed and stable behavior across documents.

**Document variability:**
- Speed variation across documents is low, indicating that model performance is robust to input differences.
- Slight dips appear for longer or denser texts, especially in CPU runs.
  
**Key insight:**
- CUDA acceleration drastically improves throughput without affecting consistency across documents.
- Llama remains the best trade-off between speed and reliability, while Phi and Gemma prioritize content quality over raw speed.

### Output outliers/hallucinations visualizations

We see the last 100 characters of each output that was truncated (df['output_tokens']>16000)

In [ ]:
N = 100
for idx, txt in enumerate(df_truncated['chat_response'].astype(str).fillna('')):
    print(f"+++++++++++++ Truncated summary {idx} (model={df_truncated.iloc[idx].get('model', '')}, system_prompt_type={df_truncated.iloc[idx].get('system_prompt_type', '')}, doc_path={df_truncated.iloc[idx].get('doc_path', '')}) +++++++++++++")
    text = txt[-N:]
    print(text)
    print("\n")